# GCM BA - 1. Regrid ERA5 onto the EC-Earth3 grid and apply the land mask

First step of the GCM bias adjustment (F1, computed at the **native EC-Earth3 resolution, 0.70 deg**):
- **ERA5** (0.25 deg) regridded **conservatively** onto the EC-Earth3 grid (Gaussian N128, 256 x 512). Conservative
  regridding uses the area overlap of the cells, so the grids do not need to be nested;
- **EC-Earth3 historical** and the **four GCM runs** (2025 / 2050, original / hosing) kept on their native grid;
- every field masked with the **LISFLOOD domain + 0.25 deg buffer on the EC-Earth3 grid** (`ecearth3_buffer`, notebook
  00c): outside the mask the values are NaN, so the BA is only meaningful (and checked) on land + coastal margin.

Periods: ERA5 and EC-Earth3 historical over `CAL_PERIOD` (1981-2014; 2005-2014 for `rsds`), the GCM runs entirely.
Units, calendar (`noleap`) and coordinates are standardised when reading (`ba_utils.read_raw`).

```
GCM_downscaling/GCM_BA/                 (EOS, next to Input_data/)
├── 1_regrid_and_mask/   <dataset>_<var>.nc : ERA5 on the EC-Earth3 grid, EC-Earth3 historical, 4 GCM runs, masked
├── 2_F1_calculation/    F1_<period>/       : WDF rule + QDM parameters
└── 3_F1_application/    F1_<period>/<run>_F1.nc : historical + 4 GCM runs, ERA5-like on the EC-Earth3 grid
```

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS_CORE = cfg["variables"]                                   # pr, tasmax, tasmin: required
VARS_ALL = VARS_CORE + cfg.get("extra_variables", [])          # + rsds where it exists
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]

CAL_PERIOD = ("1981", "2014")      # years of ERA5 / EC-Earth3 historical used for F1 (EC-Earth3 historical ends in 2014)
RSDS_FIRST_YEAR = "2005"           # ERA5 radiation starts in 2005: shorter period for rsds

def period_of(v):
    """Calibration period of a variable: CAL_PERIOD, starting in 2005 for rsds."""
    return (max(CAL_PERIOD[0], RSDS_FIRST_YEAR), CAL_PERIOD[1]) if v == "rsds" else CAL_PERIOD

IN = bu.gcm_ba_dir(cfg, "1_regrid_and_mask")                                       # prepared inputs (EOS)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}"
MASK_KIND = "buffer"     # "buffer" = ecearth3_buffer (default) | "domain" = ecearth3 (no buffer) | None = no mask
OVERWRITE = False        # False = keep the files already written
OUT = IN
print("outputs:", OUT)

## 1. EC-Earth3 grid and mask

In [ ]:
GRID = bu.read_raw(cfg, "gcm_historical", "tasmax").isel(time=0, drop=True)
for name in GCM_RUNS:
    if bu.dataset_available(cfg, name, "tasmax"):
        assert bu.same_grid(bu.read_raw(cfg, name, "tasmax").isel(time=0, drop=True), GRID), f"{name} is not on the EC-Earth3 historical grid"
if MASK_KIND is None:
    MASK = None
else:
    key = {"buffer": "ecearth3_buffer", "domain": "ecearth3"}[MASK_KIND]
    assert bu.domain_mask_path(cfg, key).exists(), f"Mask '{key}' not found: run notebook 00c_land_mask first."
    MASK = bu.load_domain_mask(cfg, key)
    assert bu.same_grid(MASK, GRID), "The mask is not on the EC-Earth3 grid: rerun notebook 00c."
    MASK = MASK.assign_coords(lat=GRID.lat, lon=GRID.lon)
print(f"EC-Earth3 grid: {GRID.sizes['lat']} x {GRID.sizes['lon']}",
      "| mask:", "none" if MASK is None else f"{key}, {int(MASK.sum()):,} of {MASK.size:,} pixels")

## 2. Regrid ERA5, mask everything, write one file per dataset and variable

In [ ]:
def write(name, var, make):
    """make() builds the field lazily (only called if the file has to be written)."""
    path = OUT / f"{name}_{var}.nc"
    if path.exists() and not OVERWRITE:
        print(f"[already done] {path.name}"); return path
    da = make()
    if MASK is not None:
        da = da.where(MASK)
    da = da.assign_coords(lat=GRID.lat, lon=GRID.lon)
    da.attrs.update({"dataset": name, "grid": "EC-Earth3 'gr' (Gaussian N128)", "mask": str(MASK_KIND)})
    print(f"-> {path.name}")
    bu.save_da(da.astype("float32").rename(var), path, name=var)
    return path


for v in VARS_ALL:
    if bu.dataset_available(cfg, "era5", v):
        write("era5", v, lambda v=v: bu.regrid_conservative(bu.subset_time(bu.read_raw(cfg, "era5", v), *period_of(v)),
                                                            GRID.lat.values, GRID.lon.values))
    else:
        print(f"[missing] ERA5 {v}")
    if bu.dataset_available(cfg, "gcm_historical", v):
        write("gcm_historical", v, lambda v=v: bu.subset_time(bu.read_raw(cfg, "gcm_historical", v), *period_of(v)))
    else:
        print(f"[missing] EC-Earth3 historical {v}")
    for name in GCM_RUNS:
        if bu.dataset_available(cfg, name, v):
            write(name, v, lambda v=v, name=name: bu.read_raw(cfg, name, v))
        else:
            print(f"[missing] {name} {v}")

## 3. Checks

- every file on exactly the EC-Earth3 grid;
- conservation of the regridding (global mean of the first 30 days, ERA5 before / after regridding, without mask);
- a look at one day: ERA5 0.25 deg and ERA5 on the EC-Earth3 grid, masked.

In [ ]:
rows = []
for f in sorted(OUT.glob("*.nc")):
    da = bu.read_processed(f)
    yrs = np.unique(da.time.dt.year.values)
    rows.append({"file": f.name, "same grid": bu.same_grid(da, GRID), "period": f"{yrs[0]}-{yrs[-1]}",
                 "n days": da.sizes["time"], "size (GB)": round(f.stat().st_size / 1e9, 2)})
files = pd.DataFrame(rows).set_index("file")
display(files)
assert files["same grid"].all(), "some files are not on the EC-Earth3 grid"

rows = []
for v in VARS_ALL:
    if not bu.dataset_available(cfg, "era5", v):
        continue
    e = bu.read_raw(cfg, "era5", v).isel(time=slice(0, 30))
    r = bu.regrid_conservative(e, GRID.lat.values, GRID.lon.values)
    a, b = float(bu.area_mean(e).mean()), float(bu.area_mean(r).mean())
    rows.append({"var": v, "ERA5 0.25 deg": round(a, 4), "on EC-Earth3 grid": round(b, 4), "difference (%)": round(100 * (b - a) / (abs(a) + 1e-9), 4)})
display(pd.DataFrame(rows).set_index("var"))

In [ ]:
v = "tasmax" if (OUT / "era5_tasmax.nc").exists() else VARS_ALL[0]
e = bu.read_raw(cfg, "era5", v).isel(time=0).compute()
r = bu.read_processed(OUT / f"era5_{v}.nc", v).isel(time=0).compute()
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
for ax, d, t in zip(axes, (e, r), (f"ERA5 0.25 deg: {v}", f"ERA5 on the EC-Earth3 grid, masked: {v}")):
    im = ax.pcolormesh(d.lon, d.lat, d, shading="auto"); plt.colorbar(im, ax=ax, shrink=0.8); ax.set_title(t)
    ax.set_facecolor("0.85")
plt.tight_layout(); plt.show()